# 029 — RAG Observability & Tracing
**Production RAG Series** | Log every hop, measure every metric, debug every failure

Covers: LangChain callbacks · Per-query tracing · Retrieval metrics · Cost/latency logging


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    return open(path, encoding='utf-8').read()

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
dl_text  = load_doc("deep_learning.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars")


✓ Loaded 22,486 chars


In [4]:
# ── 1. Why observability matters ────────────────────────────────────────
print("Common RAG failures in production:")
print()
print("  'The system gave a wrong answer'")
print("  → Was it a retrieval failure? Wrong chunks?")
print("  → Was it a generation failure? LLM hallucinated from correct context?")
print("  → Was it a query failure? Bad query embedding?")
print()
print("Without tracing you CANNOT tell which hop failed.")
print()
print("A traced RAG pipeline logs:")
print("  - Original query")
print("  - Retrieved chunks + their similarity scores")
print("  - Prompt sent to LLM (with context)")
print("  - LLM response")
print("  - Per-hop latency + total latency")
print("  - Token counts / estimated cost")


Common RAG failures in production:

  'The system gave a wrong answer'
  → Was it a retrieval failure? Wrong chunks?
  → Was it a generation failure? LLM hallucinated from correct context?
  → Was it a query failure? Bad query embedding?

Without tracing you CANNOT tell which hop failed.

A traced RAG pipeline logs:
  - Original query
  - Retrieved chunks + their similarity scores
  - Prompt sent to LLM (with context)
  - LLM response
  - Per-hop latency + total latency
  - Token counts / estimated cost


In [5]:
# ── 2. Custom RAG tracer with LangChain callbacks ───────────────────────
import time, json
from langchain.callbacks.base import BaseCallbackHandler
from typing import Any, Dict, List, Union
from langchain_core.outputs import LLMResult

class RAGTracer(BaseCallbackHandler):
    '''Callback handler that records every LLM call and chain step.'''

    def __init__(self):
        self.traces: list = []
        self._current: dict = {}
        self._t0: float = 0

    def on_llm_start(self, serialized, prompts, **kwargs):
        self._t0 = time.time()
        self._current = {
            "model": serialized.get("kwargs", {}).get("model_name", "unknown"),
            "prompt_chars": sum(len(p) for p in prompts),
        }

    def on_llm_end(self, response: LLMResult, **kwargs):
        elapsed = time.time() - self._t0
        gen = response.generations[0][0]
        self._current.update({
            "latency_s": round(elapsed, 3),
            "output_chars": len(gen.text),
            "finish_reason": getattr(gen, "generation_info", {}).get("finish_reason", "?"),
        })
        self.traces.append(dict(self._current))

    def summary(self):
        print(f"Traced {len(self.traces)} LLM call(s):")
        for i, t in enumerate(self.traces, 1):
            print(f"  [{i}] model={t.get('model','?')}  "
                  f"prompt={t.get('prompt_chars',0):,} chars  "
                  f"output={t.get('output_chars',0):,} chars  "
                  f"latency={t.get('latency_s','?')}s")

tracer = RAGTracer()
print("✓ RAGTracer callback ready")


✓ RAGTracer callback ready


In [6]:
# ── 3. Instrument the retrieval step ────────────────────────────────────
import os, numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
docs_lc = splitter.create_documents([rag_text, ml_text])
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = FAISS.from_documents(docs_lc, embeddings)

def traced_retrieve(query: str, k: int = 5) -> dict:
    t0 = time.time()
    results = vectorstore.similarity_search_with_score(query, k=k)
    elapsed = time.time() - t0
    scores = [float(s) for _, s in results]
    docs = [d for d, _ in results]
    return {
        "query": query,
        "latency_s": round(elapsed, 4),
        "n_retrieved": len(results),
        "top_score": round(min(scores), 4),   # FAISS: lower = more similar (L2)
        "avg_score": round(sum(scores) / len(scores), 4),
        "docs": docs,
        "scores": scores,
    }

trace = traced_retrieve("what is retrieval augmented generation")
print(f"Retrieval trace:")
print(f"  latency  : {trace['latency_s']}s")
print(f"  n_docs   : {trace['n_retrieved']}")
print(f"  top score: {trace['top_score']} (FAISS L2, lower = better)")
print(f"  avg score: {trace['avg_score']}")


Retrieval trace:
  latency  : 0.0071s
  n_docs   : 5
  top score: 0.6012 (FAISS L2, lower = better)
  avg score: 0.9965


In [7]:
# ── 4. Full traced RAG pipeline ─────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

RAG_PROMPT = ChatPromptTemplate.from_template(
    "Answer ONLY from the context below.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

class TracedRAGPipeline:
    def __init__(self, vectorstore, llm, embeddings):
        self.vectorstore = vectorstore
        self.llm = llm
        self.embeddings = embeddings
        self.query_log: list = []

    def run(self, question: str, k: int = 5) -> dict:
        run_start = time.time()

        # Step 1: retrieval
        ret = traced_retrieve(question, k=k)

        # Step 2: context assembly
        context = "\n\n---\n\n".join([d.page_content for d in ret["docs"]])

        # Step 3: generation with tracer callback
        tracer = RAGTracer()
        llm_traced = ChatGroq(
            model="llama-3.1-8b-instant", temperature=0, max_retries=2,
            callbacks=[tracer]
        )
        chain = RAG_PROMPT | llm_traced
        gen_start = time.time()
        response = chain.invoke({"context": context, "question": question})
        gen_latency = round(time.time() - gen_start, 3)

        total_latency = round(time.time() - run_start, 3)
        log_entry = {
            "question": question,
            "retrieval_latency_s": ret["latency_s"],
            "n_docs_retrieved": ret["n_retrieved"],
            "top_retrieval_score": ret["top_score"],
            "context_chars": len(context),
            "generation_latency_s": gen_latency,
            "total_latency_s": total_latency,
            "answer_chars": len(response.content),
            "answer_preview": response.content[:100],
        }
        self.query_log.append(log_entry)
        return log_entry

    def report(self):
        print(f"\n{'─'*60}")
        print(f"{'QUERY LOG':^60}")
        print(f"{'─'*60}")
        for i, entry in enumerate(self.query_log, 1):
            print(f"\n[{i}] Q: {entry['question'][:60]!r}")
            print(f"     ret={entry['retrieval_latency_s']}s  gen={entry['generation_latency_s']}s  "
                  f"total={entry['total_latency_s']}s")
            print(f"     docs={entry['n_docs_retrieved']}  ctx={entry['context_chars']:,}chars  "
                  f"top_score={entry['top_retrieval_score']}")
            print(f"     A: {entry['answer_preview']}...")

pipeline = TracedRAGPipeline(vectorstore, llm, embeddings)


In [8]:
# ── 5. Run queries and inspect trace ────────────────────────────────────
questions = [
    "What is retrieval augmented generation?",
    "How does BM25 differ from dense retrieval?",
    "Explain the RAPTOR summarization approach",
]

for q in questions:
    entry = pipeline.run(q)
    print(f"✓ {q[:50]!r}  total={entry['total_latency_s']}s")

pipeline.report()


✓ 'What is retrieval augmented generation?'  total=0.466s


✓ 'How does BM25 differ from dense retrieval?'  total=0.32s


✓ 'Explain the RAPTOR summarization approach'  total=0.319s

────────────────────────────────────────────────────────────
                         QUERY LOG                          
────────────────────────────────────────────────────────────

[1] Q: 'What is retrieval augmented generation?'
     ret=0.0086s  gen=0.433s  total=0.466s
     docs=5  ctx=1,472chars  top_score=0.6359
     A: Retrieval-Augmented Generation (RAG) is a hybrid AI architecture that combines information retrieval...

[2] Q: 'How does BM25 differ from dense retrieval?'
     ret=0.013s  gen=0.284s  total=0.32s
     docs=5  ctx=1,129chars  top_score=0.7191
     A: BM25 uses term-frequency methods like exact keyword matching and rare term retrieval, whereas dense ...

[3] Q: 'Explain the RAPTOR summarization approach'
     ret=0.008s  gen=0.289s  total=0.319s
     docs=5  ctx=1,648chars  top_score=0.9663
     A: RAPTOR builds a tree of summaries at multiple levels of abstraction, enabling retrieval at different...


In [9]:
# ── 6. Metrics dashboard ────────────────────────────────────────────────
import statistics

log = pipeline.query_log
if log:
    retrieval_times = [e["retrieval_latency_s"] for e in log]
    gen_times       = [e["generation_latency_s"] for e in log]
    total_times     = [e["total_latency_s"] for e in log]
    ctx_sizes       = [e["context_chars"] for e in log]

    print("PIPELINE METRICS SUMMARY")
    print("=" * 40)
    print(f"Queries run         : {len(log)}")
    print(f"Avg retrieval time  : {statistics.mean(retrieval_times):.3f}s")
    print(f"Avg generation time : {statistics.mean(gen_times):.3f}s")
    print(f"Avg total latency   : {statistics.mean(total_times):.3f}s")
    print(f"Avg context size    : {statistics.mean(ctx_sizes):,.0f} chars")
    print()
    print("Slowest query:")
    slowest = max(log, key=lambda x: x["total_latency_s"])
    print(f"  {slowest['question']!r}")
    print(f"  total={slowest['total_latency_s']}s")

# Save log to file
log_path = os.path.join(os.getcwd(), '..', 'datasets', 'rag_query_log.json')
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2)
print(f"\n✓ Query log saved to {log_path}")


PIPELINE METRICS SUMMARY
Queries run         : 3
Avg retrieval time  : 0.010s
Avg generation time : 0.335s
Avg total latency   : 0.368s
Avg context size    : 1,416 chars

Slowest query:
  'What is retrieval augmented generation?'
  total=0.466s

✓ Query log saved to /home/saurabh/projects/personal_project/advance-rag/rag_notebooks/../datasets/rag_query_log.json


In [10]:
# ── 7. Production observability tools ───────────────────────────────────
print("PRODUCTION OBSERVABILITY OPTIONS")
print("=" * 45)
print()
print("1. LANGSMITH (LangChain)")
print("   export LANGCHAIN_TRACING_V2=true")
print("   export LANGCHAIN_API_KEY=<your-key>")
print("   export LANGCHAIN_PROJECT=my-rag-project")
print("   → All chains/LLM calls auto-traced in LangSmith UI")
print()
print("2. ARIZE PHOENIX (open-source, self-hosted)")
print("   pip install arize-phoenix opentelemetry-sdk")
print("   import phoenix as px")
print("   px.launch_app()  # starts local UI on localhost:6006")
print("   from phoenix.otel import register")
print("   tracer_provider = register()")
print("   → Traces every retrieval + LLM call with full spans")
print()
print("3. CUSTOM (this notebook)")
print("   BaseCallbackHandler → log to JSON/DB → build your own dashboard")
print("   Full control, no external API keys, works offline")
print()
print("Key metrics to track in production:")
print("  retrieval_latency    context_relevance_score   answer_faithfulness")
print("  generation_latency   top_k_score               hallucination_rate")
print("  total_latency        chunk_hit_rate            user_feedback")


PRODUCTION OBSERVABILITY OPTIONS

1. LANGSMITH (LangChain)
   export LANGCHAIN_TRACING_V2=true
   export LANGCHAIN_API_KEY=<your-key>
   export LANGCHAIN_PROJECT=my-rag-project
   → All chains/LLM calls auto-traced in LangSmith UI

2. ARIZE PHOENIX (open-source, self-hosted)
   pip install arize-phoenix opentelemetry-sdk
   import phoenix as px
   px.launch_app()  # starts local UI on localhost:6006
   from phoenix.otel import register
   tracer_provider = register()
   → Traces every retrieval + LLM call with full spans

3. CUSTOM (this notebook)
   BaseCallbackHandler → log to JSON/DB → build your own dashboard
   Full control, no external API keys, works offline

Key metrics to track in production:
  retrieval_latency    context_relevance_score   answer_faithfulness
  generation_latency   top_k_score               hallucination_rate
  total_latency        chunk_hit_rate            user_feedback
